# Lab 1 - Exercise 3.1: Improving Fine-tuning Performance

Questo notebook documenta il passaggio dalla baseline head-only dell'Exercise 1.3 a un fine-tuning piu' efficace. L'obiettivo non e' provare molte idee in modo casuale, ma confrontare poche varianti motivate e discutere cosa ci dicono i risultati.

I risultati riportati qui sono quelli degli artifact della pipeline nuova in `artifacts/runs/`. Le run storiche sono conservate in `Esperimenti_di_prova.ipynb`, ma non vengono usate per trarre le conclusioni finali.


---
### Exercise 3.1 (Easy): Improving Fine-tuning Performance

In this exercise you are asked to iterate on the fine-tuning experiment performed in Exercise 1.3 in order to squeeze the best performance possible out of the model.

What can we do:

- Use a more powerful model?
- More aggressive data augmentation?
- Selective layer training?
- Something else?

L'idea scelta qui e' **selective layer training**: invece di allenare solo la `fc`, sblocchiamo anche `layer4`, cioe' l'ultimo blocco residuo della ResNet-18. E' un compromesso semplice: adattiamo le feature piu' alte al dominio GTSRB senza riaddestrare tutta la rete.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.data import build_dataloaders
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.evaluate import classification_metrics, history_to_frame, predict
from dla_lab1.models import build_classifier, count_parameters
from dla_lab1.train import resolve_device

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo training lunghi automaticamente.
RUN_TRAINING = False

print(f"Project root: {ROOT}")


Project root: c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DLA_1


## Punto di partenza: baseline Exercise 1.3

La baseline head-only e' il riferimento metodologico: ResNet-18 pre-addestrata, backbone congelato e solo classificatore finale addestrabile.

In questo notebook non copiamo valori numerici dal vecchio lavoro. Il confronto deve essere fatto con risultati prodotti dalla pipeline corrente o con artifact generati localmente.


In [2]:
baseline_context = pd.DataFrame([
    ["Exercise 1.2", "ResNet-18 feature extractor + SVM", "baseline stabile senza fine-tuning"],
    ["Exercise 1.3", "ResNet-18 frozen backbone + linear fc", "fine-tuning head-only"],
], columns=["Reference", "Setup", "Role"])

baseline_context


,Reference,Setup,Role
0,Exercise 1.2,ResNet-18 feature extractor + SVM,baseline stabile senza fine-tuning
1,Exercise 1.3,ResNet-18 frozen backbone + linear fc,fine-tuning head-only


Nota: i risultati prodotti in notebook vecchi possono aiutare a ricordare quali idee erano state provate, ma non devono essere usati come risultati finali in questa versione. Qui manteniamo solo il metodo e rilanciamo le configurazioni rilevanti nella pipeline corrente.


## Esperimenti possibili per Exercise 3.1

Le idee da provare sono tre:

- sbloccare `layer4` mantenendo congelato il resto del backbone;
- aggiungere augmentation moderata;
- combinare `layer4` trainabile e augmentation.

La tabella seguente elenca le configurazioni disponibili, senza riportare risultati copiati dal vecchio notebook.


In [ ]:
improvement_candidates = pd.DataFrame([
    ["ex3_1_layer4_unfrozen", "layer4 + fc", "none", "verifica effetto dello sblocco di layer4"],
    ["ex3_1_head_only_aggressive_aug", "fc only", "aggressive", "verifica se augmentation da sola basta"],
    ["ex3_1_layer4_aggressive_aug", "layer4 + fc", "aggressive", "augmentation forte con layer4 trainabile"],
    ["ex3_1_layer4_conservative_aug", "layer4 + fc", "conservative", "augmentation moderata con layer4 trainabile"],
    ["ex3_1_layer4_spatial_aug", "layer4 + fc", "spatial", "augmentation geometrica leggera"],
    ["ex3_1_layer4_no_aug_lr1e4_wd05", "layer4 + fc", "none", "controllo con LR piu basso e weight decay piu alto"],
    ["ex3_1_layer4_safe_aug_ls005_discriminative", "layer4 + fc", "safe", "augmentation sicura, label smoothing e LR discriminativi"],
    ["ex3_1_layer4_conservative_ls005", "layer4 + fc", "conservative", "test mirato di label smoothing su augmentation conservativa"],
], columns=["Experiment", "Trainable layers", "Augmentation", "Purpose"])

improvement_candidates


## Strategia sperimentale

La baseline head-only allena solo la `fc` finale. Se le feature ImageNet non separano bene i segnali GTSRB, questa scelta e' troppo rigida. Per questo le varianti dell'Exercise 3.1 provano a sbloccare `layer4`, cioe' l'ultimo blocco convoluzionale della ResNet-18, lasciando congelati i blocchi precedenti.

Le augmentation vengono valutate con cautela. Per GTSRB alcune trasformazioni sono semanticamente rischiose: ad esempio il flip orizzontale puo' cambiare il significato di segnali direzionali. Per questo la variante aggressiva e' utile come esperimento, ma non e' automaticamente una buona scelta finale.


## Risultati della pipeline corrente

| Run | Parametri allenati | Augmentation | Best val acc | Ultima train acc | Ultima val acc | Gap train-val finale |
|---|---|---|---:|---:|---:|---:|
| `ex1_3_head_only_adam_ce` | `fc` | none | 0.4665 | 0.8272 | 0.4653 | 0.3619 |
| `ex3_1_head_only_aggressive_aug` | `fc` | aggressive | 0.4387 | 0.6678 | 0.4296 | 0.2383 |
| `ex3_1_layer4_aggressive_aug` | `layer4` + `fc` | aggressive | 0.7433 | 0.9927 | 0.7369 | 0.2557 |
| `ex3_1_layer4_conservative_aug` | `layer4` + `fc` | conservative | 0.7541 | 0.9979 | 0.7512 | 0.2467 |
| `ex3_1_layer4_spatial_aug` | `layer4` + `fc` | spatial | 0.7089 | 0.9986 | 0.7089 | 0.2897 |
| `ex3_1_layer4_unfrozen` | `layer4` + `fc` | none | 0.7584 | 1.0000 | 0.7584 | 0.2416 |
| `ex3_1_layer4_no_aug_lr1e4_wd05` | `layer4` + `fc` | none | 0.6947 | 1.0000 | 0.6938 | 0.3062 |
| `ex3_1_layer4_safe_aug_ls005_discriminative` | `layer4` + `fc` | safe | 0.7306 | 0.9985 | 0.7282 | 0.2703 |
| `ex3_1_layer4_conservative_ls005` | `layer4` + `fc` | conservative | **0.7741** | 0.9998 | 0.7701 | 0.2297 |

Il miglior valore di validation e' ora ottenuto da `ex3_1_layer4_conservative_ls005`. Il risultato indica che la scelta piu' utile non e' stata aumentare in modo aggressivo le trasformazioni, ma regolarizzare leggermente la loss con `label_smoothing=0.05` mantenendo augmentation realistiche per i segnali stradali. La run con augmentation safe, label smoothing e learning rate discriminativi e' piu' conservativa ma non competitiva in questa configurazione; il semplice abbassamento del learning rate senza augmentation peggiora nettamente.


In [ ]:
candidate_runs = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

run_rows = []
for run_name in candidate_runs:
    summary_path = ROOT / "artifacts" / "runs" / run_name / "summary.json"
    if summary_path.exists():
        summary = pd.read_json(summary_path, typ="series")
        run_rows.append([
            run_name,
            summary.get("best_epoch", None),
            summary.get("best_val_acc", None),
            summary_path,
        ])

if run_rows:
    current_ex3_results = pd.DataFrame(
        run_rows,
        columns=["Experiment", "Best Epoch", "Best Val Accuracy", "Summary Path"],
    ).sort_values("Best Val Accuracy", ascending=False)
    display(current_ex3_results)
else:
    print("Nessun summary.json trovato per le run candidate. Esegui una run per generare risultati nel notebook corrente.")


## Configurazioni nella nuova pipeline

Le configurazioni utili per Exercise 3.1 sono ora in `config/config.yaml`. Questo permette di rilanciare le prove senza riscrivere celle lunghe nel notebook.

In [ ]:
exercise_31_experiments = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

config_rows = []
for name in exercise_31_experiments:
    exp = experiment_config(config, name)
    loss_cfg = exp["training"].get("loss_kwargs", {})
    config_rows.append([
        name,
        exp["model"]["name"],
        exp["model"]["unfreeze_layer4"],
        exp["experiment"].get("augmentation", "none"),
        exp["training"]["optimizer"],
        exp["training"]["loss"],
        exp["training"].get("label_smoothing", loss_cfg.get("label_smoothing", 0.0)),
        exp["training"]["learning_rate"],
        exp["training"].get("layer4_lr", None),
        exp["training"].get("fc_lr", None),
        exp["training"]["epochs"],
        batch_size_for(config, exp["experiment"]["batch_size_key"]),
    ])

pd.DataFrame(config_rows, columns=[
    "Experiment", "Model", "Unfreeze layer4", "Augmentation", "Optimizer",
    "Loss", "Label smoothing", "Learning rate", "Layer4 LR", "FC LR",
    "Epochs", "Batch size"
])


## Modello selezionato e overfitting

Seguendo una regola semplice e corretta, il modello da selezionare prima del test e' quello con la migliore validation accuracy: `ex3_1_layer4_conservative_ls005`. Questa scelta usa solo train e validation, quindi non contamina il test set. Il test non e' stato usato per scegliere ne' augmentation, ne' learning rate, ne' label smoothing.

Il modello continua comunque a mostrare overfitting: la train accuracy finale e' circa 99,98%, mentre la validation finale e' circa 77,01%. Il gap non implica data leakage; indica piuttosto che il modello ha capacita' sufficiente per memorizzare il training split, mentre la generalizzazione resta piu' difficile. Il label smoothing riduce leggermente il gap finale rispetto al precedente best senza augmentation e migliora la validation, ma non elimina il problema.

Cause probabili:

- GTSRB e' sbilanciato: alcune classi hanno molti piu' esempi di altre.
- Le immagini sono piccole e spesso molto simili all'interno della stessa sequenza.
- Sbloccare `layer4` aumenta molto la capacita' rispetto alla sola testa finale.
- Le augmentation aggressive non sono adatte: l'horizontal flip puo' cambiare il significato di segnali direzionali e variazioni cromatiche forti possono alterare colori semanticamente importanti.
- La validation migliora con una regolarizzazione leggera (`label_smoothing=0.05`), ma il backbone resta abbastanza potente da adattarsi quasi perfettamente al training set.


In [ ]:
SELECTED_EXPERIMENT = "ex3_1_layer4_conservative_ls005"
selected_cfg = experiment_config(config, SELECTED_EXPERIMENT)

model = build_classifier(
    model_name=selected_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=selected_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(selected_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(selected_cfg["model"].get("unfreeze_layer4", False)),
)

parameter_summary = pd.DataFrame([count_parameters(model)])
parameter_summary["trainable_ratio"] = parameter_summary["trainable"] / parameter_summary["total"]
parameter_summary

Il numero di parametri trainabili e' molto piu' alto della baseline head-only, ma resta inferiore al fine-tuning completo. Questo e' il compromesso principale dell'esercizio: piu' adattamento al dataset senza aggiornare tutta la rete.

## Cella opzionale per rilanciare il miglior esperimento

La cella seguente e' disattivata di default. Serve solo se vuoi rieseguire la variante selezionata nella nuova pipeline.

In [ ]:
if RUN_TRAINING:
    result = run_finetuning(config, SELECTED_EXPERIMENT, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare Exercise 3.1.")


## Valutazione finale sul test set del best model

Dopo aver scelto `ex3_1_layer4_conservative_ls005` sulla validation, il test set puo' essere usato una sola volta per stimare la generalizzazione finale. Questa cella e' disattivata di default per evitare di trasformare il test in un criterio di scelta. Quando la decisione e' definitiva, si imposta `RUN_FINAL_TEST_EVAL = True` e si riporta il risultato senza rilanciare altre scelte di iperparametri.


In [ ]:
RUN_FINAL_TEST_EVAL = False
FINAL_EXPERIMENT = "ex3_1_layer4_conservative_ls005"

if RUN_FINAL_TEST_EVAL:
    final_cfg = experiment_config(config, FINAL_EXPERIMENT)
    final_batch_size = batch_size_for(config, final_cfg["experiment"]["batch_size_key"])
    final_device = resolve_device(config["project"].get("device", "auto"))

    final_loaders = build_dataloaders(
        data_root=ROOT / config["paths"]["data_root"],
        image_size=int(config["dataset"]["image_size"]),
        batch_size=final_batch_size,
        val_split=float(config["dataset"]["val_split"]),
        track_size=int(config["dataset"]["track_size"]),
        seed=int(config["project"]["seed"]),
        num_workers=int(config["dataset"]["num_workers"]),
        pin_memory=bool(config["dataset"]["pin_memory"]),
        augmentation=final_cfg["experiment"].get("augmentation", "none"),
    )

    final_model = build_classifier(
        model_name=final_cfg["model"]["name"],
        num_classes=int(config["project"]["num_classes"]),
        weights=None,
        freeze_backbone=False,
    )
    checkpoint_path = ROOT / "checkpoints" / f"{FINAL_EXPERIMENT}.pt"
    final_model.load_state_dict(torch.load(checkpoint_path, map_location=final_device))

    y_true_test, y_pred_test = predict(final_model, final_loaders["test"], final_device)
    final_test_metrics = classification_metrics(y_true_test.numpy(), y_pred_test.numpy())
    print(f"Final test accuracy ({FINAL_EXPERIMENT}): {final_test_metrics['accuracy']:.4f}")
    print(final_test_metrics["classification_report"])
else:
    print(
        "Test finale non eseguito in questa apertura. "
        "Eseguirlo una sola volta dopo aver fissato il modello su validation."
    )


## Funzioni usate

Le funzioni principali sono commentate nei file `.py`:

- `experiment_config`: costruisce la configurazione completa dell'esperimento;
- `batch_size_for`: legge il batch size adatto dalla sezione hardware;
- `build_classifier`: crea ResNet-18 e sblocca `layer4` quando richiesto;
- `count_parameters`: controlla quanti parametri sono totali e trainabili;
- `run_finetuning`: esegue DataLoader, modello, training, checkpoint e artifact locali;
- `build_transforms`: applica preprocessing e augmentation scelta dal config.

## Conclusione Exercise 3.1

Il miglioramento rispetto alla baseline head-only e' netto: la validation accuracy passa da circa **46,6%** a **77,4%**. Il miglior modello osservato e' `ex3_1_layer4_conservative_ls005`, che mantiene trainabile `layer4` e la testa finale, usa augmentation conservativa e aggiunge `label_smoothing=0.05`. Questo conferma che, per GTSRB, adattare almeno l'ultimo blocco della ResNet-18 e' molto piu' efficace che allenare soltanto la testa finale.

La strategia train/validation/test e' corretta: usiamo train per aggiornare i pesi, validation per scegliere checkpoint e iperparametri, e test solo alla fine per una stima finale. Lo split train/validation e' anche piu' prudente di uno split casuale puro perche' raggruppa immagini consecutive della stessa classe, riducendo il rischio che immagini quasi identiche finiscano sia in train sia in validation.

Non vedo un problema evidente di data leakage sul test set. Il rischio principale resta l'overfitting sul training split: anche il nuovo best arriva quasi al 100% di train accuracy. La causa piu' plausibile e' la combinazione tra dataset relativamente piccolo/sbilanciato, immagini intra-classe molto simili e capacita' del blocco `layer4` sbloccato. Il label smoothing migliora la validation e riduce un po' il gap, ma non rende il modello perfettamente generalizzante.

Per migliorare ancora senza fare cross-validation, proverei in ordine:

1. class weighting o sampler bilanciato se le classi rare restano deboli nel classification report;
2. una piccola testa MLP con dropout moderato;
3. learning rate discriminativi applicati alla configurazione migliore, non alla sola variante safe;
4. una augmentation conservativa leggermente piu' ricca, sempre senza horizontal flip e senza hue shift;
5. ResNet-34 solo dopo aver stabilizzato questi aspetti su ResNet-18.

Per la consegna, il commento finale deve essere onesto: il modello migliore migliora chiaramente la baseline, ma il gap train-validation mostra che la generalizzazione non e' ancora ottimale. Il test set va valutato una sola volta dopo questa scelta, non usato per continuare a scegliere iperparametri.
